# Static arms with pfn_large: which ones are competitive, per d?

## Description

**Question.** Which fixed acquisition function (arm) works best with the frozen `pfn_large` on tasks from its own BNN prior, at d ∈ {2, 4, 6}? This is a **selection run**. It picks, per d,
- the tuned best static arm **B\*** (the baseline the switching controller must beat), and
- the switching **portfolio**: the best hyperparameter per family (ei\*, pi\*, ucb\*), plus `mes`, `es` and `mean`.

The switching experiment (`switching`) then re-measures all of these on fresh tasks (split 1). Its baseline is therefore not a best-of-13 number from these tasks: the selection effect stays here.

**Setup.**
- **Tasks.** 200 functions per d, drawn from the BNN prior pfn_large was trained on (tuning split 0, seeds `bo/tasks.py::task_seed`). f\* comes from a dense Sobol search (16,384 points) plus Adam refinement, and is lowered to the best value any arm finds.
- **Budget.** n_init = 2d shared scrambled-Sobol points, then T = 10d BO steps. The context reaches 12d ≤ 72 < 100, which is inside pfn_large's training range.
- **Candidates (CRN).** Each (task, step) gets the same shifted Sobol set of 1024 points, plus 3 × 64 incumbent perturbations. Every arm on a task scores the same candidates whenever the histories agree.
- **Arms.**
  - `ei`, `ei0.01`, `ei0.05` (EI margin on the ECDF scale);
  - `pi0.01`, `pi0.05`, `pi0.1`;
  - `ucb0.8` … `ucb0.99` (the quantile of the PFN predictive);
  - `mean` (pure exploitation);
  - `mes` (max-value entropy search on the bar distribution);
  - `es` (amortized entropy search: the alpha-PFN `alpha_pes_large`);
  - and random search.
- **Numerics.** fp32 throughout (TF32 off; log §1).

**What to look for.**
- **Metric.** Mean over tasks of log10(final simple regret + 1e-4), with bootstrap CIs over tasks. Regret curves show the shape over steps.
- **Expected pattern (prior work, pfn-acqpolicy §16).** EI is near the top at d = 2 and under-explores at d ≥ 4, where UCB-type arms should gain.
- **Paired differences** to the selected B\* indicate how flat the top is. A flat top means switching among the top arms has little to gain.

**Selection rule (fixed before the run).**
- Per d, **B\*** is the arm with the lowest mean final log10 regret. Random search is excluded.
- Per family (ei, pi, ucb), the hyperparameter with the lowest mean is selected.
- **Portfolio** = [B\*, then the rest of {ei\*, pi\*, ucb\*, mes, es, mean} without B\*]. That is 6 arms, with B\* first, because ties in the controller go to the first arm.

**GO / NO-GO.** This is a selection run, so GO means "the selection is made and every arm ran on all tasks". NO-GO would mean a broken arm, for example worse than random search at every d, which would point to a bug and not to a result. No scientific claim is made from these tasks.

**Limits.**
- BNN-prior tasks only: in-distribution for the PFN by construction. This is the setting where an amortized policy would be trained, and the most favourable one for it.
- Noise-free observations.
- A single nuisance seed. The task-to-task spread dominates the spread from that seed.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/static_tune/<sweep_id>"  # set after `just snapshot runs/static_tune/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.